## 0.1 Prepare dataset

In [ ]:
import requests
import tarfile
from pathlib import Path
import shutil
import tempfile

# Dataset URL
dataset_url = "http://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz"

# Use path for temporary files, then copy to data directory
tmp_dir = Path(tempfile.gettempdir())
dataset_file = tmp_dir / "booksummaries.tar.gz"

# Path to data directory
data_dir = Path.cwd()
target_file = data_dir / "booksummaries.txt"

if not target_file.exists():
    print(f"Downloading CMU Book Summary Dataset from {dataset_url}...")

    # Download the file to /tmp
    response = requests.get(dataset_url, stream=True)
    response.raise_for_status()

    dataset_file.write_bytes(response.content)
    print(f"Downloaded to {dataset_file}")

    # Extract the archive
    print("Extracting archive...")
    with tarfile.open(dataset_file, "r:gz") as tar:
        tar.extractall(path=tmp_dir, filter='data')

    # Copy file to data directory
    extracted_file = tmp_dir / "booksummaries" / "booksummaries.txt"
    shutil.copy(str(extracted_file), str(target_file))

    # Cleanup
    shutil.rmtree(tmp_dir / "booksummaries")
    dataset_file.unlink()

    print(f"✓ Dataset ready: {target_file}")
else:
    print(f"✓ Dataset already exists: {target_file}")

## 1. Imports

In [ ]:
# Optional cuda install
%pip install torch --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%pip install bertopic
%pip install bertopic[visualization]
%pip install pandas scikit-learn nltk
%pip install sentence-transformers umap-learn plotly

In [ ]:
import pandas as pd
from bertopic import BERTopic
import nltk
from sklearn.feature_extraction.text import CountVectorizer
import os
import numpy as np
from umap import UMAP

nltk.download('stopwords')
nltk.download('punkt')  # for tokenization
nltk.download('wordnet')  # contains word lemmas
nltk.download('omw-1.4')  # for lemmatization
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import torch

## 2. Loading and preliminary data cleaning

We load the dataset and remove rows that do not contain plot summaries. We also drop duplicate book titles to ensure each book is unique.

In [66]:
file_path = 'booksummaries.txt'

if not os.path.exists(file_path):
    print(f"File {file_path} not foud.")
else:
    column_names = ['wiki_id', 'freebase_id', 'book_title', 'author', 'publication_date', 'genres', 'plot_summary']
    df = pd.read_csv(file_path, sep='\t', header=None, names=column_names)

    df.dropna(subset=['plot_summary'], inplace=True)
    df.drop_duplicates(subset=['book_title'], inplace=True)

    print(f"Loaded and processed {len(df)} books.")
    print("Sample Data:")
    display(df.head())

Loaded and processed 16277 books.
Sample Data:


,wiki_id,freebase_id,book_title,author,publication_date,genres,plot_summary
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,986,/m/0ldx,The Plague,Albert Camus,1947,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...
3,1756,/m/0sww,An Enquiry Concerning Human Understanding,David Hume,NaN,NaN,The argument of the Enquiry proceeds by a ser...
4,2080,/m/0wkt,A Fire Upon the Deep,Vernor Vinge,NaN,"{""/m/03lrw"": ""Hard science fiction"", ""/m/06n90...",The novel posits that space around the Milky ...


### 2.1 Better data cleaning

In [67]:
import re
import html


def clean_summary_text(text):
    """
    Cleans the plot summary text by removing HTML tags, wiki markup, and other unwanted patterns.
    """
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = re.sub(r'==+.*?==+', ' ', text)
    text = re.sub(r'~Plot outline description~', '', text, flags=re.IGNORECASE)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\{\{.*?\}\}', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\[\[(?:[^\|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


df['plot_summary'] = df['plot_summary'].apply(clean_summary_text)

num_books_before = len(df)
print(f"Number of books before cleaning: {num_books_before}")

MIN_LENGTH_THRESHOLD = 500

is_valid_summary = df['plot_summary'].str.len() >= MIN_LENGTH_THRESHOLD
removed_books = df[~is_valid_summary]
df = df[is_valid_summary].reset_index(drop=True)
num_books_after = len(df)
print(
    f"Number of books after cleaning: {num_books_after} (removed {num_books_before - num_books_after} books with short summaries)")

if not removed_books.empty:
    print("\nExamples of removed books due to short summaries:")
    display(removed_books[['book_title', 'plot_summary']].head(10))

Number of books before cleaning: 16277
Number of books after cleaning: 13744 (removed 2533 books with short summaries)

Examples of removed books due to short summaries:


,book_title,plot_summary
8,Blade Runner 3: Replicant Night,"Living on Mars, Deckard is acting as a consult..."
48,Icehenge,"Icehenge is part mystery, part psychological d..."
137,Mutiny on the Bounty,The novel tells the story through a fictional ...
138,The Mothman Prophecies,The book combines Keel's account of his invest...
228,East Lynne,"Lady Isabel Carlyle, a beautiful and refined y..."
314,The Hustler,"After losing to Fats, Eddie could spiral down ..."
338,Born Yesterday,"An uncouth, corrupt rich junk dealer, Harry Br..."
363,The Whalestoe Letters,Pelafina writes these letters to Johnny from T...
365,Mister Roberts,"The title character, a Lieutenant Junior Grade..."
387,Science and Health with Key to the Scriptures,Science and Health encapsulates the teachings ...


## 3. Generating embeddings
Generating Embeddings with Sentence Transformers, more specifically the 'all-mpnet-base-v2' model. It provides 768-dimensional embeddings.
It will run on GPU if available, otherwise on CPU.

In [68]:
from sentence_transformers import SentenceTransformer

if 'df' in locals():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)
    summaries = df['plot_summary'].tolist()

    print(f"Starting to generate embeddings for {len(summaries)} books...")
    embeddings = model.encode(
        summaries,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=64
    )
    print(f"Created embeddings with shape: {embeddings.shape}")
else:
    print("No data present")

Using device: cuda
Starting to generate embeddings for 13744 books...


Batches: 100%|██████████| 215/215 [01:24<00:00,  2.53it/s]

Created embeddings with shape: (13744, 768)


Embeddings can be saved to a file for later use, mainly for backend.

In [27]:
embeddings_file = 'book_embeddings.npy'

print("Saving embeddings to file...")
np.save(embeddings_file, embeddings)

print(f"Embeddings saved to {embeddings_file}. Data shape: {embeddings.shape}")

Saving embeddings to file...
Embeddings saved to book_embeddings.npy. Data shape: (15193, 768)


## 4. Topic Modeling with BERTopic
BERTopic enables us to discover topics within the book summaries. For example, we might find topics related to "romance", "science fiction", "historical events", etc.

### Text Preprocessing

We create a function to clean the plot summaries: we remove special characters, stop-words (commonly occurring words like 'the' and 'a') and perform lemmatization (reducing words to their base form).

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    text = text.lower()

    # Removing special characters and digits
    text = re.sub(r'\W|\d', ' ', text)

    tokens = nltk.word_tokenize(text)

    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)


if 'df' in locals():
    print("Beginning text processing...")
    df['processed_summary'] = df['plot_summary'].apply(preprocess_text)
    print("Text processing finished.")

    print("\n--- Example --- ")
    print("Original summary:")
    print(df.iloc[0]['plot_summary'][:500])
    print("\nSummary after processing:")
    print(df.iloc[0]['processed_summary'][:500])
else:
    print("No data present")

### Training BERTopic

BERTopic will automatically extract topics from the processed summaries. `CountVectorizer` is used to remove English stop-words during the creation of the topic representation.

In [81]:
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

if 'df' in locals():
    documents = df['plot_summary'].tolist()

    custom_stopwords = [
        "book", "novel", "story", "tale",
        "chapter", "author", "writes", "written"
    ]

    stopwords = list(ENGLISH_STOP_WORDS) + custom_stopwords

    vectorizer_model = CountVectorizer(
        stop_words=stopwords,
        min_df=2,
    )

    representation_model = KeyBERTInspired()

    topic_umap = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine', random_state=42)

    topic_model = BERTopic(
        embedding_model=model,
        umap_model=topic_umap,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        min_topic_size=30,
        top_n_words=10,
        verbose=True

    )
    topics, probabilities = topic_model.fit_transform(documents, embeddings)

    print("\nModel has been trained successfully!")
    print(topic_model.get_topic_info())
else:
    print("No data present")

2025-12-04 23:01:09,844 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-04 23:01:15,625 - BERTopic - Dimensionality - Completed ✓
2025-12-04 23:01:15,626 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-04 23:01:15,904 - BERTopic - Cluster - Completed ✓
2025-12-04 23:01:15,907 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-04 23:01:22,070 - BERTopic - Representation - Completed ✓



Model has been trained successfully!
    Topic  Count                                            Name  \
0      -1   5985                -1_learns_attack_discovers_death   
1       0   1420                       0_town_death_tells_things   
2       1    467                         1_lord_death_david_dies   
3       2    439            2_war_destroy_civilization_destroyed   
4       3    418                   3_tells_alice_children_george   
5       4    407                    4_battle_begins_death_castle   
6       5    349              5_death_narrator_character_suicide   
7       6    349            6_god_machiavelli_nietzsche_religion   
8       7    311                7_narrator_death_tells_discovers   
9       8    294                       8_war_narrator_death_dies   
10      9    249               9_pirates_village_odysseus_pompey   
11     10    226  10_protagonist_characters_character_revolution   
12     11    216           11_narrator_suspects_suspect_murderer   
13     12 

### Visualizing and Analyzing Topics

**Most frequent topics**

In [82]:
if 'topic_model' in locals():
    display(topic_model.get_topic_info())
else:
    print("No df_topic")

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5985,-1_learns_attack_discovers_death,"[learns, attack, discovers, death, murder, tow...","[Los Angeles, 1936 Kay Fischer is an architect..."
1,0,1420,0_town_death_tells_things,"[town, death, tells, things, named, realizes, ...","[In the novel Tricks, the story begins with fi..."
2,1,467,1_lord_death_david_dies,"[lord, death, david, dies, wife, husband, affa...","[The narrative opens with Mr Bingley, a wealth..."
3,2,439,2_war_destroy_civilization_destroyed,"[war, destroy, civilization, destroyed, like, ...",[The story begins on a human colony world of V...
4,3,418,3_tells_alice_children_george,"[tells, alice, children, george, father, pinoc...","[;""His Mate"" A rough collie named Lad lives at..."
5,4,407,4_battle_begins_death_castle,"[battle, begins, death, castle, journey, gate,...",[The novel begins with the Companions assemble...
6,5,349,5_death_narrator_character_suicide,"[death, narrator, character, suicide, killed, ...","[Milada Kralicek, a young Czechoslovakian girl..."
7,6,349,6_god_machiavelli_nietzsche_religion,"[god, machiavelli, nietzsche, religion, descri...",[Dawkins dedicates the book to Douglas Adams a...
8,7,311,7_narrator_death_tells_discovers,"[narrator, death, tells, discovers, raoul, soc...",[The novel recounts the experiences of the Nar...
9,8,294,8_war_narrator_death_dies,"[war, narrator, death, dies, house, father, li...",[The story revolves around many characters rep...


**Visualization of topics as a bar chart**

This chart shows the 10 most popular topics and the keywords that define them.

In [83]:
if 'topic_model' in locals():
    display(topic_model.visualize_barchart(top_n_topics=10))
else:
    print("No df_topic")

**Map of distances between topics**

It shows topics as bubbles. The size of a bubble corresponds to its popularity. Bubbles located close to each other are thematically similar. It help to understand the relationships between topics.

In [84]:
if 'topic_model' in locals():
    display(topic_model.visualize_topics())
else:
    print("No df_topic")

**Hierarchical clustering of topics**

Shows how topics can be grouped into larger clusters. Useful for identifying overarching themes in literature.

In [85]:
if 'topic_model' in locals():
    display(topic_model.visualize_hierarchy())
else:
    print("No df_topic")

### Saving the Model

We save the trained model for easy reuse later. We also save the processed data for use in an API.

In [87]:
if 'topic_model' in locals():
    model_dir = 'model'
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model_path = os.path.join(model_dir, 'bertopic_model')
    topic_model.save(model_path, serialization='safetensors')

    print(f"Model has been saved in: {model_path}")

Model has been saved in: model\bertopic_model


## 5. Dimensionality Reduction

### PCA

### UMAP

In [ ]:
# Optional embeddings loading from file
embeddings_file = 'book_embeddings.npy'

print("Loading embeddings from file...")
embeddings = np.load(embeddings_file)
print(f"Loaded embeddings with shape: {embeddings.shape}")

In [ ]:
if 'embeddings' in locals():
    reducer = UMAP(
        n_neighbors=15,
        n_components=2,  # 2d
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )

    print("Starting dimensionality reduction (UMAP)...")
    umap_embeddings = reducer.fit_transform(embeddings)
    print(f"New shape after reduction: {umap_embeddings.shape}")

In [ ]:
#optional reduced embeddings saving
umap_embeddings_file = 'umap_embeddings.npy'

print("Saving embeddings 2d to file...")
np.save(umap_embeddings_file, umap_embeddings)

print(f"Embeddings saved to {umap_embeddings_file}. Data shape: {umap_embeddings.shape}")

### PCA + UMAP

## 6. Saving book data

## 7. Visualization with Plotly

In [ ]:
# Genre extraction for better testing

import ast


def genres_to_readable(genres_str):
    """Converts the genre JSON/dict string into a human-readable list of genres."""
    try:
        genres_dict = ast.literal_eval(str(genres_str))
        if isinstance(genres_dict, dict) and genres_dict:
            return ", ".join(genres_dict.values())
    except Exception:
        pass
    return "Unknown"


print("Processing genres...")
df['genres_readable'] = df['genres'].apply(genres_to_readable)

df.head()

In [ ]:
import plotly.express as px

if 'df' in locals():
    df_transformers = df.copy()

    df_transformers['x'] = umap_embeddings[:, 0]
    df_transformers['y'] = umap_embeddings[:, 1]

    print("Generating interactive plot...")

    fig = px.scatter(
        df_transformers,
        x='x',
        y='y',
        color='main_genre',
        hover_data=['book_title', 'author', 'main_genre', 'genres_readable'],
        title="Book similarity map",
        template='plotly_dark'
    )

    fig.update_traces(marker=dict(size=4, opacity=0.7))
    fig.update_layout(legend_title_text='Main Genre')

    fig.show()

## 8. Adding a new book

In [ ]:
new_book = {
    'title': "Martyr!",
    'author': "Kaveh Akbar",
    'plot_summary': """
Cyrus is a queer poet living in Indiana, recovering from addiction to alcohol and drugs. His father, now deceased, was an Iranian migrant worker on a farm in rural Indiana. Cyrus believes that when he was a baby, his mother was killed on Iran Air Flight 655, a passenger plane which was shot down by a US missile during the Iran-Iraq war.

Cyrus is interested in the idea of martyrdom, and begins working on a "book of martyrs", while considering his own conceptual suicide as a potential martyr. He hears of an Iranian performance artist named Orkideh, who has terminal breast cancer, and is spending her last days in the Brooklyn Museum as part of a Marina Abramović-esque performance piece named "Death-Speak". Cyrus travels to New York to talk with Orkideh, bringing his roommate Zee. Zee has strong feelings for Cyrus, although their relationship is often fraught.

The book cycles between time periods and perspectives, including chapters told by Cyrus' mother, who was in a secret lesbian relationship in Iran, and his uncle, who, while serving in the Iran-Iraq war, was instructed to dress as an "angel" on horseback in order to comfort dying Iranian soldiers on the battlefield.

Upon Orkideh's death, Cyrus finds out from Sang - Orkideh's gallerist and ex-wife - that Orkideh was actually his mother, Roya. Roya had swapped passports with her lover Leila in order to escape Iran, and Leila was killed on the plane. In a dreamlike final scene, Cyrus appears to reconcile with Zee, before walking into a pool of golden light.
    """
}

new_embedding = model.encode([new_book['plot_summary']])

new_point = reducer.transform(new_embedding)

fig.add_scatter(
    x=[new_point[0, 0]],
    y=[new_point[0, 1]],
    mode="markers+text",
    text=["New Book"],
    marker=dict(size=12, color="red", symbol="x"),
    name="New Book"
)
fig.show()